[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/spin_transformations/spin_transformations.ipynb)

# Spin Transformations

This example computes spin transformations: bendings of a surface that keep every angle. A quaternion at each vertex, a turn and a scale, acts on the edges around it, and [Crane, Pinkall and Schröder (2011)](https://www.cs.cmu.edu/~kmcrane/Projects/SpinTransformations/paper.pdf) choose the quaternions so that the surface's mean curvature changes by a prescribed amount. Repeated, the transformation becomes a conformal curvature flow that rounds Spot the cow into a sphere; asked for a constant change, its fields are the spinor spherical harmonics, and they spin the sphere into Dirac spheres.

In the algebra of three-dimensional space the quaternions are the even multivectors, and the vertices and edges are points, antivectors, intrinsic like all geometry, which in three dimensions are bivectors. Every operator is a sparse extensor, a linear map between fields on the vertices, edges and faces of a mesh in which each element couples to only a few others by a multivector: the Dirac operator couples each face to its corners by bivectors, so it takes quaternion fields to quaternion fields, and the energy of a quaternion field, its area-weighted squared residual, `~residual * face_areas * residual`, couples vertices by quaternions. The new edges come from sandwiching each old edge with the quaternions along it, `quaternions << edges`.

In the quaternionic notation of Crane, Pinkall and Schröder the quaternions are the solution $\lambda$ of the eigenvalue problem $(D - \rho)\,\lambda = \gamma\lambda$ of least $|\gamma|$, and each new edge is $\tilde e = \bar\lambda\, e\, \lambda$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga.algebras import VGA3D as ga
from numga.sparse import SparseExtensor, spdiag
from examples import instantiate
from examples.util.animation import save_figure, show_animation
from examples.surfaces.spin_transformations import render
from examples.surfaces.spin_transformations.core import spot

# Meshes and their incidences as sparse maps between fields, in the algebra of space.
mesh_module = instantiate("examples.util.mesh", ga)
Mesh, as_ga_sparse, as_scalar = mesh_module.Mesh, mesh_module.as_ga_sparse, mesh_module.as_scalar
mv = mesh_module.ctx.multivector
Scalar = ga.gatype.scalar()
# Vertices and edges are points, antivectors, intrinsic like all geometry; quaternions are the even multivectors.
Point = ga.gatype.antivector()
Vector = ga.gatype.vector()
Even = ga.gatype.even()

np.set_printoptions(precision=4, suppress=True)

# A unit sphere, its icosahedron split so many times.
n_subdivisions = 3
mesh, vertices = Mesh.icosphere(n_subdivisions)
print(vertices.gatype, mesh.edges.shape, mesh.faces.shape)

## 1. Operators between fields

A field is an extensor with one element per vertex, edge or face, such as `vertices`, `Point[V]`. An operator between fields is a sparse extensor, a linear map that couples each element of one field to a few elements of another, each coupling an extensor. Here the couplings are multivectors, nullary extensors, which act by their product: times a field, each coupling multiplies its input element into its output element, and the products are summed; times another operator it composes the two; and `~` is its reverse, with inputs and outputs swapped and every coupling reversed. The mesh's incidences say which elements couple. These operators couple through scalars.

The geometry is read off the vertices. The edges are `boundary * vertices`, and the edge facing each corner of each face is one of them, picked out by the face's indices and signed as the face runs along it. A face's area is half the size of the plane two of its edges span. An edge's cotangent weight is half the cotangents of the two angles facing it, each cotangent one edge's slice read along the other, over the size of the plane they span. From these come the diagonal maps of the face areas, of the vertex areas, a third of each face at each of its corners, and of the cotangent weights; the cotangent Laplacian, `~boundary * edge_weights * boundary`; and `unit_volume`, the sense the counter-clockwise faces give space, plus or minus one.

In [ ]:
def cotangent_weights(mesh: Mesh, sides: Point) -> Scalar:
    """`Scalar[E]`: each edge's cotangent weight from the edge facing each corner `[3] Point[F]`, half the cotangents
    of the angles facing it; a corner's cotangent is one edge's slice read along the other, over the size of the
    plane they span."""
    after, before = sides[[1, 2, 0]], sides[[2, 0, 1]]                         # [3] Point[F] each
    cotangents = -(after.dual_inverse() & before) / after.commutator(before).norm()   # [3] Scalar[F]
    return ~mesh.by_corner(cotangents / 2) * as_scalar(np.ones(len(mesh.faces))).field()


def sense_of_space(mesh: Mesh, vertices: Point) -> Scalar:
    """The sense the counter-clockwise faces give space, plus or minus one: the faces' corners meeting about the
    origin, summed over the solid, over its size."""
    a, b, c = vertices.batch()[mesh.faces.T].field()                           # Point[F] each
    volume = (a & b & c).sites.sum()                                           # [] Scalar
    return volume / volume.norm()

In [ ]:
# The boundary, taking each edge's tail from its head, and the means over each edge and each face.
boundary = mesh.d0                                                             # [E, V] Scalar
edge_means = as_ga_sparse(mesh.edges, as_scalar(np.ones_like(mesh.edges) / 2))  # [E, V] Scalar
face_means = as_ga_sparse(mesh.faces, as_scalar(np.ones_like(mesh.faces) / 3))  # [F, V] Scalar

edges = boundary * vertices                                                    # Point[E]
sides = (edges.batch()[mesh.face_edges.T] * mesh.face_edge_orientation.T).field()   # [3] Point[F]
areas = sides[1].commutator(sides[0]).norm() / 2                               # Scalar[F]
face_areas, vertex_areas = spdiag(areas), spdiag(~face_means * areas)          # [F, F], [V, V] Scalar
edge_weights = spdiag(cotangent_weights(mesh, sides))                          # [E, E] Scalar
laplacian = ~boundary * edge_weights * boundary                                # [V, V] Scalar
unit_volume = sense_of_space(mesh, vertices)                                   # [] Scalar

In [ ]:
print(boundary.gatype, boundary.shape)
print(edges.gatype)

## 2. A change in mean curvature

The transformation is asked for by a change in each face's mean curvature, `curvature_change`, a scalar field on the faces. Here it grows along y, each face's centroid paired with the direction y, `centroids & mv.y`: the surface is to bend more on one side and less on the other, red where it is to bend more.

In [ ]:
# How strongly the mean curvature changes along y.
dipole = 0.5

In [ ]:
centroids = face_means * vertices                                       # Point[F]
curvature_change = (centroids & mv.y) * dipole                          # Scalar[F]

save_figure(render.draw_surface(mesh, vertices, curvature_change), "spin_transformations_curvature");

## 3. The Dirac operator

The Dirac operator sends a quaternion field on the vertices to a field on the faces. Crane, Pinkall and Schröder build it edge by edge: each face takes the quaternion at each corner by the edge that corner faces, over twice the face's area, so the cells of `dirac` are bivectors and it takes a quaternion field to a quaternion field on the faces. The same operator is the geometric derivative, with each face's unit normal multiplied in on the right. On each face a field on the vertices is linear in between, and `derivative` takes it to its gradient there: the coupling to each corner is the gradient of that corner's hat function, the edge the corner faces paired with the face's unit normal, `edge | normal`, the edge turned a quarter turn within the face, over twice the face's area. `derivative * field` is the field's geometric derivative, and with the normal on the right it is `dirac * field`. The potential puts the curvature change, read against the sense the surface's counter-clockwise faces give space, `unit_volume`, on each face's mean, a third at each corner. The transformation's quaternions solve `dirac * field == potential * field`, so the residual couples each face to each corner by the edge the corner faces over twice the area, less a third of the change: its cells are even, and it takes a field to how far it is from a transformation. The energy of a field is its area-weighted squared residual, `~residual * face_areas * residual`, its cells quaternions too.

In the quaternionic matrix notation of Crane, Pinkall and Schröder this reads as $D_{ij} = -\tfrac{1}{2\mathcal{A}_i} e_j$, with $e_j$ the edge as an imaginary quaternion, minus its bivector in Hamilton's convention, $A = D - R$, and the energy $A^{H} M_F A$.

In [ ]:
# Each face takes each corner by the edge it faces, over twice the face's area; with the face's normal, the same
# couplings are the gradients of the corners' hat functions.
dirac = mesh.at_corners(sides / (2 * areas))                                    # [F, V] Point
face_normals = (sides[0] & sides[1]) / (2 * areas)                              # Vector[F]
derivative = mesh.at_corners((sides | face_normals) / (2 * areas))              # [F, V] Vector
# The residual of `dirac * field == potential * field`, and its area-weighted square.
residual = mesh.at_corners(sides / (2 * areas) - curvature_change * unit_volume / 3)   # [F, V] Even
energy = ~residual * face_areas * residual                                      # [V, V] Even

In [ ]:
print(derivative.gatype, dirac.gatype)
print(energy.gatype, energy.shape)

## 4. The quaternion field

The quaternions are the field of least energy per unit of vertex area, the least eigenpair of `energy` against the vertex areas. An eigensolve has to know what a linear map acts on, so `energy * Even` and `vertex_areas * Even` turn its multivectors into maps on quaternions. The field is fixed only up to a constant factor, which would turn and scale the whole surface, so it is divided by its area-weighted mean. The colours show how much each face's mean quaternion scales it, red where it grows and blue where it shrinks.

In [ ]:
# The least eigenfield of the energy against the vertex areas, divided by its area-weighted mean.
_, (field,) = (energy * Even).eigh(vertex_areas * Even, 1)                     # Even[V]
quaternions = field / ((vertex_areas * field).sites.sum() / areas.sites.sum())  # Even[V]
scaling = (face_means * quaternions).scalar_norm_squared() - 1                  # Scalar[F]

save_figure(render.draw_surface(mesh, vertices, scaling), "spin_transformations_scaling");

## 5. The new surface

Each edge is turned and scaled by the field along it, the reverse sandwich `quaternion << edge` with the quaternion running linearly from one end to the other, integrated by Simpson's rule: the turns at the two ends and twice the turn at the middle. The turns at the ends are averaged as maps, `quaternions << Point`, the turn at each vertex as a map on points, so that `edge_means` carries them to the edges before they meet the edge they turn; the turn at the middle is the sandwich of the mean quaternion. The new vertices are those whose edges match the turned ones best, weighted by the cotangent Laplacian: the least-squares solution of the Laplacian acting on points, `laplacian * Point`, which leaves the surface centred. The checkerboard is the sphere's: a conformal map keeps its squares square.

In the quaternionic notation of Crane, Pinkall and Schröder this reads as $\tilde e_{ij} = \tfrac13 \bar\lambda_i e_{ij} \lambda_i + \tfrac16 \bar\lambda_i e_{ij} \lambda_j + \tfrac16 \bar\lambda_j e_{ij} \lambda_i + \tfrac13 \bar\lambda_j e_{ij} \lambda_j$.

In [ ]:
# Simpson's rule along each edge: the turns at its two ends, averaged as maps, and twice the turn at its middle.
moved = ((edge_means * (quaternions << Point))(edges) + 2 * ((edge_means * quaternions) << edges)) / 3   # Point[E]
deformed = (laplacian * Point).lstsq(~boundary * edge_weights * moved)          # Point[V]

save_figure(render.draw_textured(mesh, deformed, vertices), "spin_transformations_deformed");

## 6. Conformal curvature flow

Repeating the transformation, each time asking for a share of every face's departure from the mean curvature to be taken away, rounds a surface step by step while every step keeps its angles. Spot the cow, by Keenan Crane, draws in her legs, horns and ears and rounds into a sphere; a checkerboard laid on the cow keeps its squares square all the way, since no step changes an angle.

In [ ]:
# The share of each face's curvature departure each step removes, the steps, and the duration of a frame.
rate = 0.2
n_flow_steps = 15
duration_ms = 160

In [ ]:
def recentred(vertices: Point) -> Point:
    """The vertices moved to their centre and scaled to unit mean distance from it: what a spin transformation leaves
    free."""
    vertices = vertices - vertices.sites.mean()
    return vertices / vertices.norm().sites.mean()


def rounding(mesh: Mesh, vertices: Point, n_steps: int, rate: float):
    """The vertices at the start and after each step that takes away a share `rate` of every face's departure from
    the area-weighted mean curvature, a change in curvature, by one spin transformation."""
    edge_means = as_ga_sparse(mesh.edges, as_scalar(np.ones_like(mesh.edges) / 2))   # [E, V] Scalar
    face_means = as_ga_sparse(mesh.faces, as_scalar(np.ones_like(mesh.faces) / 3))   # [F, V] Scalar
    vertices = recentred(vertices)
    yield vertices
    for _ in range(n_steps):
        # The operators of the surface as it stands.
        edges = mesh.d0 * vertices                                             # Point[E]
        sides = (edges.batch()[mesh.face_edges.T] * mesh.face_edge_orientation.T).field()   # [3] Point[F]
        areas = sides[1].commutator(sides[0]).norm() / 2                       # Scalar[F]
        face_areas, vertex_areas = spdiag(areas), spdiag(~face_means * areas)  # [F, F], [V, V] Scalar
        edge_weights = spdiag(cotangent_weights(mesh, sides))                  # [E, E] Scalar
        laplacian = ~mesh.d0 * edge_weights * mesh.d0                          # [V, V] Scalar
        unit_volume = sense_of_space(mesh, vertices)                           # [] Scalar
        # Each face's signed mean curvature: the cotangent Laplacian of the positions is the area-integrated
        # mean-curvature normal at each vertex, met with the vertex's normal, read against the sense the faces give
        # space, over twice the vertex's area, and averaged onto the faces.
        normals = (~face_means * (sides[0] & sides[1])).normalized()            # Vector[V]
        curvature = face_means * (((laplacian * vertices) & normals) * unit_volume / (2 * (~face_means * areas)))   # Scalar[F]
        change = -rate * (curvature - (curvature * areas).sites.sum() / areas.sites.sum())   # Scalar[F]
        # The spin transformation of that change, as above.
        residual = mesh.at_corners(sides / (2 * areas) - change * unit_volume / 3)   # [F, V] Even
        _, (field,) = ((~residual * face_areas * residual) * Even).eigh(vertex_areas * Even, 1)   # Even[V]
        quaternions = field / ((vertex_areas * field).sites.sum() / areas.sites.sum())   # Even[V]
        moved = ((edge_means * (quaternions << Point))(edges) + 2 * ((edge_means * quaternions) << edges)) / 3   # Point[E]
        vertices = recentred((laplacian * Point).lstsq(~mesh.d0 * edge_weights * moved))
        yield vertices


cow, cow_vertices = spot()
show_animation(render.animate_rounding(cow, rounding(cow, cow_vertices, n_flow_steps, rate)), "spin_rounding", duration_ms)

## 7. Dirac spheres

With `rho` a constant `n`, the fields of zero energy are the eigenfields of the Dirac operator itself. Their eigenvalues are the integers except −1, each with multiplicity one more than itself, and spinning the sphere by one gives a Dirac sphere. `dirac_spheres` runs the energy above with that constant and spins the sphere by each of the fields of least energy, one row per eigenvalue. The higher states are finer, so the sphere is split once more.

Crane, Pinkall and Schröder name the physics these fields come from (§6.2):

> Eigenfunctions of D on the unit sphere are called the spinor spherical harmonics, which can be used to compute fundamental solutions to the Dirac equation for the configuration of a spin-1/2 particle such as an electron in a spherically symmetric potential.

In [ ]:
# The finer sphere's subdivisions, the eigenvalues shown, and the states shown of each.
n_dirac_subdivisions = 4
eigenvalues = (1, 2, 3, -2, -3)
n_states = 3

In [ ]:
def dirac_spheres(mesh: Mesh, vertices: Point, eigenvalues: np.ndarray, n_states: int) -> tuple[Scalar, Point]:
    """For each constant curvature change, the eigenvalues `[eigenvalues]`, the energies of the `n_states` fields of
    least energy, and the vertices of the surfaces they spin the mesh into, `[eigenvalues, states] Point[V]`; the
    fields at unit size per unit of vertex area, as they come."""
    edge_means = as_ga_sparse(mesh.edges, as_scalar(np.ones_like(mesh.edges) / 2))   # [E, V] Scalar
    face_means = as_ga_sparse(mesh.faces, as_scalar(np.ones_like(mesh.faces) / 3))   # [F, V] Scalar
    edges = mesh.d0 * vertices                                                 # Point[E]
    sides = (edges.batch()[mesh.face_edges.T] * mesh.face_edge_orientation.T).field()   # [3] Point[F]
    areas = sides[1].commutator(sides[0]).norm() / 2                           # Scalar[F]
    face_areas, vertex_areas = spdiag(areas), spdiag(~face_means * areas)      # [F, F], [V, V] Scalar
    edge_weights = spdiag(cotangent_weights(mesh, sides))                      # [E, E] Scalar
    laplacian = ~mesh.d0 * edge_weights * mesh.d0                              # [V, V] Scalar

    # Every eigenvalue's constant change at once, a batch of residuals.
    constant = as_scalar(eigenvalues[:, None]) * sense_of_space(mesh, vertices) / 3   # [eigenvalues, 1] Scalar
    residual = mesh.at_corners(sides / (2 * areas) - constant)                 # [eigenvalues] [F, V] Even
    energies, fields = ((~residual * face_areas * residual) * Even).eigh(vertex_areas * Even, n_states)   # [eigenvalues, states] Even[V]
    moved = ((edge_means * (fields << Point))(edges) + 2 * ((edge_means * fields) << edges)) / 3   # [eigenvalues, states] Point[E]
    return energies, (laplacian * Point).lstsq(~mesh.d0 * edge_weights * moved)


fine, fine_vertices = Mesh.icosphere(n_dirac_subdivisions)
dirac_energies, spun = dirac_spheres(fine, fine_vertices, np.array(eigenvalues, dtype=float), n_states)   # [eigenvalues, states]

save_figure(render.draw_dirac(fine, dict(zip(eigenvalues, spun)), fine_vertices), "dirac_spheres");

In [ ]:
# checks
# Crane's Dirac operator is the geometric derivative with each face's unit normal on the right, where it
# anticommutes with the in-plane gradients; and the geometric derivative of position is twice the volume
# element on every face: the dimension of the surface, with no curl.
np.testing.assert_allclose((spdiag(face_normals) * derivative + dirac).cells.kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((derivative * vertices).kernel, np.broadcast_to([0, 0, 0, 2], (len(mesh.faces), 4)), atol=1e-12)
# Every corner keeps its angle, up to the mesh's resolution: one edge's slice read along the other, over both
# lengths, is its cosine.
def corner_cosines(vertices: Point) -> Scalar:
    sides = ((boundary * vertices).batch()[mesh.face_edges.T] * mesh.face_edge_orientation.T).field()   # [3] Point[F]
    after, before = sides[[1, 2, 0]], sides[[2, 0, 1]]                         # [3] Point[F] each
    return -(after.dual_inverse() & before) / (after.norm() * before.norm())


np.testing.assert_allclose((corner_cosines(deformed) - corner_cosines(vertices)).to_array(), 0.0, atol=0.05)
# The Dirac spheres' fields have zero energy, up to the mesh's resolution against the spectrum's unit spacing.
np.testing.assert_allclose(dirac_energies.to_array(), 0.0, atol=0.05)